In [ ]:
# Run once
!pip install sentence-transformers datasets scipy scikit-learn pandas tqdm umap-learn

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import json
import umap
from sklearn.decomposition import PCA
from scipy import stats
import matplotlib.pyplot as plt

In [ ]:
# Load data into a pandas DataFrame
import pandas as pd
df = pd.read_csv('../data/WVS Statements.csv')
df

In [ ]:
#model_name = "intfloat/multilingual-e5-large-instruct"
#model_name = "all-MiniLM-L6-v2"
#model_name = "allenai/specter"
#model_name = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
model_name = "BAAI/bge-base-en"

# Load model
embedding_model = SentenceTransformer(model_name)

In [ ]:
# Select one question as a test
df_test = df[df['WVS question'] == 'Q90']
df_test

In [ ]:
# Compute embeddings for the selected question
embeddings = embedding_model.encode(df_test['Statement'].tolist())

In [ ]:
# Initialize PCA
pca = PCA(n_components=1, random_state=42)

# Fit PCA to embeddings and transform the data
pca_result = pca.fit_transform(embeddings)

In [ ]:
# Create a DataFrame for plotting from PCA results and add metadata
pca_df = pd.DataFrame(data = pca_result, columns = ['Embedded value'])
pca_df['text'] = df_test['Statement'].reset_index(drop=True)
pca_df['Target value'] = df_test['Code'].reset_index(drop=True)

In [ ]:
# Normalize column 'Embedded value' values to minimize the distance with 'Target value' values using a linear regression
ratio, constant, r_value, p_value, std_err = stats.linregress(pca_df['Embedded value'], pca_df['Target value'])
pca_df['Normalized embedded value'] = ratio * pca_df['Embedded value'] + constant

# Delete 'Embedded value' column
pca_df = pca_df.drop(columns=['Embedded value'])
pca_df.head()

In [ ]:
# Plot result
plt.figure(figsize=(10, 9))
plt.scatter(pca_df['Target value'], pca_df['Normalized embedded value'], alpha=0.7)

# Add labels to a subset of points to avoid overlap
# Select points at regular intervals (every Nth point)
n = len(pca_df)
label_every = max(1, n // 15)  # Label approximately 15 points

for i in range(0, n, label_every):
    plt.annotate(pca_df.iloc[i]['text'], 
                 (pca_df.iloc[i]['Target value'], pca_df.iloc[i]['Normalized embedded value']),
                 xytext=(5, 5), textcoords='offset points', 
                 fontsize=8, alpha=0.8,
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='none'))

plt.title('Normalized embedded value vs target value (ground truth) for '+model_name)
plt.xlabel('Target Value')
plt.ylabel('Normalized Embedded Value')
plt.grid()
plt.show()

In [ ]:
# Compute the spearman correlation
correlation, p_value = stats.spearmanr(pca_df['Target value'], pca_df['Normalized embedded value'])
print(f"Spearman correlation: {correlation:.4f}, p-value: {p_value:.4f}")